Gestión de datos y procesamiento de información
Evaluación 2 Caso 2
Catalina Jiménez Lillo
Docente: Natalia Morales Rosales
18/06/2026

Para este análisis se usó el archivo `walmart.xlsx`, el dataset utilizado para el Caso 2 fue obtenido desde Kaggle.La base contiene ventas realizadas en Estados Unidos entre 2011 y 2014.
Tiene 3.203 registros y 12 columnas, con información sobre pedidos, clientes, productos, ventas y ganancias.
En total, hay 1.611 pedidos, 686 clientes, 169 ciudades, 11 estados, 17 categorías y 1.494 productos.
La variable Country contiene únicamente el valor United States, por lo que no es útil realizar comparaciones entre países.

Objetivos:
El objetivo de este caso es limpiar y preparar la base de ventas de Walmart para responder preguntas relacionadas con:
- Los estados con mayores ventas y ganancias.
- Las categorías más vendidas y rentables.
- La evolución de las ventas a través de los años.
- Los productos que generan más ingresos.
- Los clientes que más gastan.
- Las ventas rentables y las que generan pérdidas.
- El tiempo promedio de envío.

1.¿En qué estados se vende más y se obtienen mejores ganancias?
2.¿Qué categorías se venden más y cuáles son las más rentables?
3.¿Las ventas y ganancias han ido mejorando con el tiempo, o más bien han bajado?
4.¿Qué productos son los que más plata dejan?
5.¿Qué clientes son los que más gastan en total?
6.¿Cuántas ventas fueron rentables, generaron pérdidas o tuvieron ganancia neutra?
7.¿Cuál es el tiempo promedio transcurrido entre la fecha del pedido y la fecha de envío?

EXTRACT: extracción de datos

In [166]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path
from sqlalchemy import create_engine, text

In [167]:
# Definir la ruta del archivo Excel
ruta_excel = Path("walmart.xlsx")

In [168]:
# Verificar que el archivo exista
ruta_excel.exists()

True

In [169]:
# Revisar las hojas disponibles en el Excel
archivo_excel = pd.ExcelFile(ruta_excel)
archivo_excel.sheet_names

['Walmart']

In [170]:
# Cargar la hoja Walmart
Walmart = pd.read_excel(ruta_excel, sheet_name="Walmart")

In [171]:
# Visualizar los primeros registros
Walmart.head()

,Order ID,Order Date,Ship Date,Customer Name,Country,City,State,Category,Product Name,Sales,Quantity,Profit
0,CA-2013-138688,2013-06-13,2013-06-17,Darrin Van Huff,United States,Los Angeles,California,Labels,Self-Adhesive Address Labels for Typewriters b...,14.620,2,6.8714
1,CA-2011-115812,2011-06-09,2011-06-14,Brosina Hoffman,United States,Los Angeles,California,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.860,7,14.1694
2,CA-2011-115812,2011-06-09,2011-06-14,Brosina Hoffman,United States,Los Angeles,California,Art,Newell 322,7.280,4,1.9656
3,CA-2011-115812,2011-06-09,2011-06-14,Brosina Hoffman,United States,Los Angeles,California,Phones,Mitel 5320 IP Phone VoIP phone,907.152,4,90.7152
4,CA-2011-115812,2011-06-09,2011-06-14,Brosina Hoffman,United States,Los Angeles,California,Binders,DXL Angle-View Binders with Locking Rings by S...,18.504,3,5.7825


TRANSFORM: limpieza y creación de variables

In [172]:
# Revisar las dimensiones del dataset
print("Tabla Walmart:", Walmart.shape)

Tabla Walmart: (3203, 12)


In [173]:
# Revisar los nombres originales de las columnas
Walmart.columns

Index(['Order ID', 'Order Date', 'Ship Date', 'Customer Name', 'Country',
       'City', 'State', 'Category', 'Product Name', 'Sales', 'Quantity',
       'Profit'],
      dtype='object')

In [174]:
# Revisar los tipos de datos y valores no nulos
Walmart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3203 entries, 0 to 3202
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Order ID       3203 non-null   object        
 1   Order Date     3203 non-null   datetime64[ns]
 2   Ship Date      3203 non-null   datetime64[ns]
 3   Customer Name  3203 non-null   object        
 4   Country        3203 non-null   object        
 5   City           3203 non-null   object        
 6   State          3203 non-null   object        
 7   Category       3203 non-null   object        
 8   Product Name   3203 non-null   object        
 9   Sales          3203 non-null   float64       
 10  Quantity       3203 non-null   int64         
 11  Profit         3203 non-null   float64       
dtypes: datetime64[ns](2), float64(2), int64(1), object(7)
memory usage: 300.4+ KB


In [175]:
# Revisar los valores nulos
Walmart.isna().sum()

Order ID         0
Order Date       0
Ship Date        0
Customer Name    0
Country          0
City             0
State            0
Category         0
Product Name     0
Sales            0
Quantity         0
Profit           0
dtype: int64

In [176]:
# Eliminar registros duplicados
Walmart = Walmart.drop_duplicates()

In [177]:
# Revisar las dimensiones después de eliminar duplicados
print("Walmart sin duplicados:", Walmart.shape)

Walmart sin duplicados: (3203, 12)


In [178]:
# Obtener estadísticas descriptivas
Walmart.describe()

,Order Date,Ship Date,Sales,Quantity,Profit
count,3203,3203,3203.000000,3203.000000,3203.000000
mean,2013-05-10 03:06:07.530440192,2013-05-14 01:25:25.195129600,226.493233,3.828910,33.849032
min,2011-01-07 00:00:00,2011-01-09 00:00:00,0.990000,1.000000,-3399.980000
25%,2012-05-22 00:00:00,2012-05-26 00:00:00,19.440000,2.000000,3.852000
50%,2013-07-22 00:00:00,2013-07-25 00:00:00,60.840000,3.000000,11.166400
75%,2014-05-23 00:00:00,2014-05-27 00:00:00,215.809000,5.000000,33.000400
max,2014-12-31 00:00:00,2015-01-06 00:00:00,13999.960000,14.000000,6719.980800
std,NaN,NaN,524.876877,2.260947,174.109081


In [179]:
# Limpiar y estandarizar los nombres de las columnas
Walmart.columns = [
    unicodedata.normalize("NFKD", columna)
    .encode("ascii", "ignore")
    .decode("utf-8")
    for columna in Walmart.columns
]

Walmart.columns = (
    pd.Index(Walmart.columns)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace(".", "_", regex=False)
)

In [180]:
# Revisar los nombres de las columnas después de la limpieza
Walmart.columns

Index(['order_id', 'order_date', 'ship_date', 'customer_name', 'country',
       'city', 'state', 'category', 'product_name', 'sales', 'quantity',
       'profit'],
      dtype='object')

In [181]:
# Convertir las columnas al formato fecha
Walmart["order_date"] = pd.to_datetime(Walmart["order_date"],errors="coerce")
Walmart["ship_date"] = pd.to_datetime(Walmart["ship_date"],errors="coerce")

In [182]:

Walmart[["order_date","ship_date"]].head()

,order_date,ship_date
0,2013-06-13,2013-06-17
1,2011-06-09,2011-06-14
2,2011-06-09,2011-06-14
3,2011-06-09,2011-06-14
4,2011-06-09,2011-06-14


In [183]:

Walmart = Walmart.drop_duplicates()

In [184]:

print("Walmart sin duplicados:", Walmart.shape)

Walmart sin duplicados: (3203, 12)


In [185]:
# Completar los valores nulos de las columnas de texto
Walmart["customer_name"] = Walmart["customer_name"].fillna("Sin información")
Walmart["country"] = Walmart["country"].fillna("Sin información")
Walmart["city"] = Walmart["city"].fillna("Sin información")
Walmart["state"] = Walmart["state"].fillna("Sin información")
Walmart["category"] = Walmart["category"].fillna("Sin información")
Walmart["product_name"] = Walmart["product_name"].fillna("Sin información")

In [186]:
# Normalizar los valores de texto
Walmart["customer_name"] = Walmart["customer_name"].astype(str).str.lower().str.strip()
Walmart["country"] = Walmart["country"].astype(str).str.lower().str.strip()
Walmart["city"] = Walmart["city"].astype(str).str.lower().str.strip()
Walmart["state"] = Walmart["state"].astype(str).str.lower().str.strip()
Walmart["category"] = Walmart["category"].astype(str).str.lower().str.strip()
Walmart["product_name"] = Walmart["product_name"].astype(str).str.lower().str.strip()

In [187]:
# Revisar los valores existentes en la columna
Walmart["product_name"].value_counts()

product_name
staples                                                 60
avery non-stick binders                                  8
bretford rectangular conference table tops               7
global troy executive leather low-back tilter            7
safco arco folding chair                                 7
                                                        ..
avery 505                                                1
rca visys 25423re1 corded phone                          1
master giant foot doorstop, safety yellow                1
bush westfield collection bookcases, fully assembled     1
avaya 5410 digital phone                                 1
Name: count, Length: 1494, dtype: int64

In [188]:
Walmart["customer_name"].value_counts()

customer_name
william brown       24
arthur prichep      23
rick wilson         19
greg guthrie        17
zuschuss carroll    16
                    ..
ross devincentis     1
benjamin venier      1
nick radford         1
monica federle       1
victor preis         1
Name: count, Length: 686, dtype: int64

In [189]:
Walmart["country"].value_counts()

country
united states    3203
Name: count, dtype: int64

In [190]:
Walmart["city"].value_counts()

city
los angeles       747
san francisco     510
seattle           428
san diego         170
phoenix            63
                 ... 
cheyenne            1
yucaipa             1
redding             1
citrus heights      1
layton              1
Name: count, Length: 169, dtype: int64

In [191]:
Walmart["state"].value_counts()

state
california    2001
washington     506
arizona        224
colorado       182
oregon         124
utah            53
nevada          39
new mexico      37
idaho           21
montana         15
wyoming          1
Name: count, dtype: int64

In [192]:
Walmart["category"].value_counts()

category
binders        471
paper          450
furnishings    304
phones         277
storage        266
accessories    258
art            250
chairs         207
appliances     136
labels         116
tables         116
bookcases       80
fasteners       72
supplies        69
envelopes       67
machines        39
copiers         25
Name: count, dtype: int64

In [193]:
Walmart.isna().sum()

order_id         0
order_date       0
ship_date        0
customer_name    0
country          0
city             0
state            0
category         0
product_name     0
sales            0
quantity         0
profit           0
dtype: int64

In [194]:
# Calcular los días transcurridos entre el pedido y el envío
Walmart["dias_envio"] = (
    Walmart["ship_date"] - Walmart["order_date"]
    ).dt.days

In [195]:
Walmart[["order_date","ship_date","dias_envio"]].head()

,order_date,ship_date,dias_envio
0,2013-06-13,2013-06-17,4
1,2011-06-09,2011-06-14,5
2,2011-06-09,2011-06-14,5
3,2011-06-09,2011-06-14,5
4,2011-06-09,2011-06-14,5


In [196]:
# Crear la variable año del pedido
Walmart["anio_pedido"] = Walmart["order_date"].dt.year

In [197]:
Walmart[["order_date","anio_pedido"]].head()

,order_date,anio_pedido
0,2013-06-13,2013
1,2011-06-09,2011
2,2011-06-09,2011
3,2011-06-09,2011
4,2011-06-09,2011


In [198]:
# Crear la variable mes del pedido
Walmart["mes_pedido"] = Walmart["order_date"].dt.month

In [199]:
Walmart[["order_date","mes_pedido"]].head()

,order_date,mes_pedido
0,2013-06-13,6
1,2011-06-09,6
2,2011-06-09,6
3,2011-06-09,6
4,2011-06-09,6


In [200]:
# Calcular el margen de ganancia
Walmart["margen_ganancia"] = (Walmart["profit"]/ Walmart["sales"].replace(0, np.nan))

In [201]:
Walmart[["sales","profit","margen_ganancia"]].head()

,sales,profit,margen_ganancia
0,14.620,6.8714,0.4700
1,48.860,14.1694,0.2900
2,7.280,1.9656,0.2700
3,907.152,90.7152,0.1000
4,18.504,5.7825,0.3125


In [202]:
# Clasificar las ventas según la ganancia obtenida
Walmart["clasificacion_ganancia"] = np.select(
    [
        Walmart["profit"] > 0,
        Walmart["profit"] < 0
    ],
    [
        "Rentable",
        "Pérdida"
    ],
    default="Neutro"
)

In [203]:
Walmart[["sales","profit","clasificacion_ganancia"]].head()

,sales,profit,clasificacion_ganancia
0,14.620,6.8714,Rentable
1,48.860,14.1694,Rentable
2,7.280,1.9656,Rentable
3,907.152,90.7152,Rentable
4,18.504,5.7825,Rentable


In [204]:
# Clasificar las ventas según su monto
Walmart["categoria_venta"] = pd.cut(
    Walmart["sales"],
    bins=[0,50,200,float("inf")],
    labels=["Baja","Media","Alta"]
)

In [205]:
Walmart[["sales","categoria_venta"]]

,sales,categoria_venta
0,14.620,Baja
1,48.860,Baja
2,7.280,Baja
3,907.152,Alta
4,18.504,Baja
...,...,...
3198,36.240,Baja
3199,91.960,Media
3200,258.576,Alta
3201,29.600,Baja


In [206]:
# Calcular el monto de venta por unidad
Walmart["venta_por_unidad"] = Walmart["sales"]/ Walmart["quantity"].replace(0, np.nan)

In [207]:
Walmart[["sales","quantity","venta_por_unidad"]].head()

,sales,quantity,venta_por_unidad
0,14.620,2,7.310
1,48.860,7,6.980
2,7.280,4,1.820
3,907.152,4,226.788
4,18.504,3,6.168


In [208]:
# Crear la tabla final del proceso ETL
ventas_final = Walmart[
    [
        "order_id",
        "order_date",
        "ship_date",
        "dias_envio",
        "anio_pedido",
        "mes_pedido",
        "customer_name",
        "country",
        "city",
        "state",
        "category",
        "product_name",
        "sales",
        "quantity",
        "profit",
        "venta_por_unidad",
        "margen_ganancia",
        "clasificacion_ganancia",
        "categoria_venta"
    ]
]

In [209]:
ventas_final.head()

,order_id,order_date,ship_date,dias_envio,anio_pedido,mes_pedido,customer_name,country,city,state,category,product_name,sales,quantity,profit,venta_por_unidad,margen_ganancia,clasificacion_ganancia,categoria_venta
0,CA-2013-138688,2013-06-13,2013-06-17,4,2013,6,darrin van huff,united states,los angeles,california,labels,self-adhesive address labels for typewriters b...,14.620,2,6.8714,7.310,0.4700,Rentable,Baja
1,CA-2011-115812,2011-06-09,2011-06-14,5,2011,6,brosina hoffman,united states,los angeles,california,furnishings,eldon expressions wood and plastic desk access...,48.860,7,14.1694,6.980,0.2900,Rentable,Baja
2,CA-2011-115812,2011-06-09,2011-06-14,5,2011,6,brosina hoffman,united states,los angeles,california,art,newell 322,7.280,4,1.9656,1.820,0.2700,Rentable,Baja
3,CA-2011-115812,2011-06-09,2011-06-14,5,2011,6,brosina hoffman,united states,los angeles,california,phones,mitel 5320 ip phone voip phone,907.152,4,90.7152,226.788,0.1000,Rentable,Alta
4,CA-2011-115812,2011-06-09,2011-06-14,5,2011,6,brosina hoffman,united states,los angeles,california,binders,dxl angle-view binders with locking rings by s...,18.504,3,5.7825,6.168,0.3125,Rentable,Baja


In [210]:
# Crear el dataset analítico
dataset_analitico = (
    ventas_final
    .groupby(["anio_pedido","mes_pedido","state","category"],as_index=False )
    .agg(
        cantidad_registros=("order_id", "count"),
        unidades_vendidas=("quantity", "sum"),
        total_ventas=("sales", "sum"),
        total_ganancia=("profit", "sum")
    )
    .sort_values("total_ventas",ascending=False)
)

dataset_analitico

,anio_pedido,mes_pedido,state,category,cantidad_registros,unidades_vendidas,total_ventas,total_ganancia
1077,2014,3,washington,copiers,1,4,13999.960,6719.9808
128,2011,7,california,supplies,3,14,8243.870,343.8098
971,2013,12,california,tables,6,29,6603.008,-327.1793
1322,2014,10,california,binders,9,45,5481.544,2048.9550
1224,2014,8,california,appliances,6,31,5143.560,1393.3963
...,...,...,...,...,...,...,...,...
332,2012,1,colorado,binders,1,2,1.938,-1.3566
882,2013,10,arizona,art,1,1,1.408,0.1584
238,2011,10,washington,binders,1,1,1.344,0.4704
510,2012,9,colorado,binders,1,2,1.080,-0.7920


In [211]:
# Revisar las dimensiones de la tabla final
ventas_final.shape

(3203, 19)

In [212]:
# Revisar los valores nulos de la tabla final
ventas_final.isna().sum()

order_id                  0
order_date                0
ship_date                 0
dias_envio                0
anio_pedido               0
mes_pedido                0
customer_name             0
country                   0
city                      0
state                     0
category                  0
product_name              0
sales                     0
quantity                  0
profit                    0
venta_por_unidad          0
margen_ganancia           0
clasificacion_ganancia    0
categoria_venta           0
dtype: int64

In [213]:
# Revisar las estadísticas de la tabla final
ventas_final.describe()

,order_date,ship_date,dias_envio,anio_pedido,mes_pedido,sales,quantity,profit,venta_por_unidad,margen_ganancia
count,3203,3203,3203.000000,3203.000000,3203.000000,3203.000000,3203.000000,3203.000000,3203.000000,3203.000000
mean,2013-05-10 03:06:07.530440192,2013-05-14 01:25:25.195129600,3.930066,2012.729941,8.025289,226.493233,3.828910,33.849032,60.724548,0.219487
min,2011-01-07 00:00:00,2011-01-09 00:00:00,0.000000,2011.000000,1.000000,0.990000,1.000000,-3399.980000,0.540000,-2.100000
25%,2012-05-22 00:00:00,2012-05-26 00:00:00,3.000000,2012.000000,5.000000,19.440000,2.000000,3.852000,6.336000,0.100000
50%,2013-07-22 00:00:00,2013-07-25 00:00:00,4.000000,2013.000000,9.000000,60.840000,3.000000,11.166400,18.336000,0.290000
75%,2014-05-23 00:00:00,2014-05-27 00:00:00,5.000000,2014.000000,11.000000,215.809000,5.000000,33.000400,63.966000,0.375000
max,2014-12-31 00:00:00,2015-01-06 00:00:00,7.000000,2014.000000,12.000000,13999.960000,14.000000,6719.980800,3499.990000,0.500000
std,NaN,NaN,1.806914,1.138640,3.254793,524.876877,2.260947,174.109081,132.665099,0.279832


In [214]:
# Revisar los tipos de datos de la tabla final
ventas_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3203 entries, 0 to 3202
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   order_id                3203 non-null   object        
 1   order_date              3203 non-null   datetime64[ns]
 2   ship_date               3203 non-null   datetime64[ns]
 3   dias_envio              3203 non-null   int64         
 4   anio_pedido             3203 non-null   int32         
 5   mes_pedido              3203 non-null   int32         
 6   customer_name           3203 non-null   object        
 7   country                 3203 non-null   object        
 8   city                    3203 non-null   object        
 9   state                   3203 non-null   object        
 10  category                3203 non-null   object        
 11  product_name            3203 non-null   object        
 12  sales                   3203 non-null   float64 

LOAD: carga a PostgreSQL

In [215]:
# Definir los datos de conexión a PostgreSQL
usuario = "postgres"
password = "bobby2405"
host = "localhost"
puerto = "1234"
base_datos = "Walmart"

In [216]:
# Crear la conexión con PostgreSQL
engine = create_engine(
 f"postgresql+psycopg2://{usuario}:{password}@{host}:{puerto}/{base_datos}"
)

In [217]:
# Probar que la conexión funcione correctamente
with engine.connect() as conexion:
 resultado = conexion.execute(text("SELECT version();"))
 print(resultado.fetchone())

('PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35225, 64-bit',)


In [218]:
# Cargar la tabla final
ventas_final.to_sql(
 name="ft_ventas_walmart",
 con=engine,
 schema="public",
 if_exists="replace",
 index=False
)


203

In [219]:
print("Tabla de Walmart cargada correctamente en PostgreSQL")

Tabla de Walmart cargada correctamente en PostgreSQL


Consultas SQL de análisis

In [220]:
# Consultar los resultados generales de las ventas
consulta = """
SELECT 
    COUNT(*) AS total_registros,
    SUM(sales) AS total_ventas,
    SUM(profit) AS total_ganancia
FROM public.ft_ventas_walmart;
"""

In [221]:
validacion = pd.read_sql(consulta,engine)
validacion

,total_registros,total_ventas,total_ganancia
0,3203,725457.8245,108418.4489


In [222]:
# Consultar las ventas y ganancias por estado
consulta = """
SELECT
    state,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    SUM(quantity) AS unidades_vendidas,
    ROUND(SUM(sales)::numeric, 2) AS total_ventas,
    ROUND(SUM(profit)::numeric, 2) AS total_ganancia
FROM public.ft_ventas_walmart
GROUP BY state
ORDER BY total_ventas DESC;
"""

In [223]:
ventas_estado = pd.read_sql(consulta,engine)
ventas_estado

,state,pedidos_distintos,unidades_vendidas,total_ventas,total_ganancia
0,california,1021,7665.0,457687.63,76381.39
1,washington,256,1883.0,138641.27,33402.65
2,arizona,108,862.0,35282.00,-3427.92
3,colorado,79,693.0,32108.12,-6527.86
4,oregon,56,499.0,17431.15,-1190.47
5,nevada,23,168.0,16729.10,3316.77
6,utah,26,219.0,11220.06,2546.53
7,montana,8,56.0,5589.35,1833.33
8,new mexico,22,151.0,4783.52,1157.12
9,idaho,11,64.0,4382.49,826.72


In [224]:
# Consultar los resultados por categoría
consulta = """
SELECT
    category,
    SUM(quantity) AS cantidad_vendida,
    SUM(sales) AS total_ventas,
    SUM(profit) AS total_ganancia
FROM public.ft_ventas_walmart
GROUP BY category
ORDER BY total_ventas DESC;
"""

In [225]:
ventas_categoria = pd.read_sql(consulta,engine)
ventas_categoria

,category,cantidad_vendida,total_ventas,total_ganancia
0,chairs,734.0,101781.3280,4027.5843
1,phones,1066.0,98684.3520,9110.7426
2,tables,481.0,84754.5620,1482.6073
3,storage,1039.0,70532.8520,8645.3222
4,accessories,1032.0,61114.1160,16484.5983
5,binders,1868.0,55961.1130,16096.8016
6,copiers,88.0,49749.2420,19327.2351
7,machines,147.0,42444.1220,-618.9264
8,bookcases,306.0,36004.1235,-1646.5117
9,appliances,492.0,30236.3360,8261.2699


In [226]:
# Consultar la evolución anual de las ventas
consulta = """
SELECT
    anio_pedido,
    SUM(sales) AS total_ventas,
    SUM(profit) AS total_ganancia
FROM public.ft_ventas_walmart
GROUP BY anio_pedido
ORDER BY anio_pedido ASC;
"""

In [227]:
ventas_anuales = pd.read_sql(consulta,engine)
ventas_anuales

,anio_pedido,total_ventas,total_ganancia
0,2011,147883.0330,20065.6912
1,2012,139966.2495,20492.1947
2,2013,186976.0165,23959.9374
3,2014,250632.5255,43900.6256


In [228]:
# Consultar los productos con mayores ventas
consulta = """
SELECT
    product_name,
    SUM(quantity) AS cantidad_vendida,
    SUM(sales) AS total_ventas,
    SUM(profit) AS total_ganancia
FROM public.ft_ventas_walmart
GROUP BY product_name
ORDER BY total_ventas DESC
LIMIT 10;
"""

In [229]:
productos_mayores_ventas = pd.read_sql(consulta, engine)
productos_mayores_ventas

,product_name,cantidad_vendida,total_ventas,total_ganancia
0,canon imageclass 2200 advanced copier,4.0,13999.960,6719.9808
1,high speed automatic electric letter opener,8.0,13100.240,524.0096
2,global troy executive leather low-back tilter,25.0,10019.600,626.2250
3,fellowes pb500 electric punch plastic comb bin...,8.0,8134.336,3050.3760
4,gueststacker chair with chrome finish legs,27.0,8030.016,803.0016
5,okidata mb760 printer,7.0,7834.400,881.3700
6,bretford rectangular conference table tops,26.0,7710.665,180.5424
7,logitechâ p710e mobile speakerphone,29.0,7467.210,1418.7699
8,canon pc1060 personal laser copier,12.0,6719.904,2267.9676
9,hewlett packard laserjet 3310 copier,13.0,6239.896,2183.9636


In [230]:
# Consultar los clientes con mayores compras
consulta = """
SELECT
    customer_name,
    SUM(quantity) AS cantidad_comprada,
    SUM(sales) AS total_compras
FROM public.ft_ventas_walmart
GROUP BY customer_name
ORDER BY total_compras DESC
LIMIT 10;
"""

In [231]:
clientes_mayores_compras = pd.read_sql(consulta,engine)
clientes_mayores_compras

,customer_name,cantidad_comprada,total_compras
0,raymond buch,23.0,14345.2760
1,ken lonsdale,32.0,8472.3940
2,edward hooks,51.0,7447.7700
3,jane waco,35.0,7391.5300
4,karen ferguson,34.0,7182.7660
5,nick crebassa,38.0,6734.2330
6,clay ludtke,63.0,6069.6440
7,yana sorensen,38.0,5754.1720
8,nora preis,42.0,5564.5985
9,william brown,93.0,5523.0540


In [232]:
# Consultar la clasificación de las ventas
consulta = """
SELECT
    clasificacion_ganancia,
    COUNT(*) AS cantidad_ventas,
    SUM(sales) AS total_ventas,
    SUM(profit) AS total_ganancia
FROM public.ft_ventas_walmart
GROUP BY clasificacion_ganancia
ORDER BY cantidad_ventas DESC;
"""

In [233]:
clasificacion_resultados = pd.read_sql(consulta,engine)
clasificacion_resultados

,clasificacion_ganancia,cantidad_ventas,total_ventas,total_ganancia
0,Rentable,2863,641746.6235,131139.4098
1,Pérdida,318,74925.2990,-22720.9609
2,Neutro,22,8785.9020,0.0000


In [234]:
# Consultar el tiempo promedio de envío
consulta = """
SELECT
    AVG(dias_envio) AS promedio_dias_envio,
    MIN(dias_envio) AS minimo_dias_envio,
    MAX(dias_envio) AS maximo_dias_envio
FROM public.ft_ventas_walmart;
"""

In [235]:
tiempo_envio = pd.read_sql(consulta,engine)
tiempo_envio

,promedio_dias_envio,minimo_dias_envio,maximo_dias_envio
0,3.930066,0,7


In [236]:
# Consultar las ventas que generaron pérdidas
consulta = """
SELECT
    order_id,
    customer_name,
    state,
    category,
    product_name,
    sales,
    profit
FROM public.ft_ventas_walmart
WHERE profit < 0
ORDER BY profit ASC
LIMIT 20;
"""

In [237]:
ventas_con_perdida = pd.read_sql(consulta,engine)
ventas_con_perdida

,order_id,customer_name,state,category,product_name,sales,profit
0,CA-2014-134845,sharelle roach,colorado,machines,lexmark mx611dhe monochrome laser printer,2549.985,-3399.9800
1,US-2013-157490,laurel beltran,colorado,machines,zebra gk420t direct thermal/thermal transfer p...,703.710,-938.2800
2,CA-2013-109869,tanja norvell,arizona,tables,bush advantage collection racetrack conference...,1272.630,-814.4832
3,US-2012-103471,jim radford,colorado,bookcases,"atlantic metals mobile 4-shelf bookcases, cust...",590.058,-786.7440
4,CA-2011-148383,resi pã¶lking,arizona,binders,gbc docubind 300 electric binding machine,946.764,-694.2936
5,CA-2014-159282,gary hansen,arizona,machines,swingline sm12-08 microcut jam free shredder,599.985,-479.9880
6,US-2013-100839,noah childs,colorado,tables,hon 5100 series wood tables,727.450,-465.5680
7,US-2011-169789,maureen fritzler,arizona,binders,ibico ibimaster 300 manual binding system,551.985,-459.9875
8,US-2014-142573,maris laware,arizona,tables,"chromcraft 48"" x 96"" racetrack double pedestal...",801.600,-448.8960
9,CA-2013-109827,laurel workman,arizona,machines,panasonic kx mc6040 color laser multifunction ...,269.970,-386.9570


In [238]:
# Crear una tabla resumen por categoría
resumen_categoria = (
    ventas_final
    .groupby("category", as_index=False)
    .agg(
        total_ventas_categoria=("sales", "sum"),
        total_ganancia_categoria=("profit", "sum"),
        unidades_vendidas_categoria=("quantity", "sum")
    )
    .sort_values("total_ventas_categoria",ascending=False)
)

resumen_categoria

,category,total_ventas_categoria,total_ganancia_categoria,unidades_vendidas_categoria
5,chairs,101781.3280,4027.5843,734
13,phones,98684.3520,9110.7426,1066
16,tables,84754.5620,1482.6073,481
14,storage,70532.8520,8645.3222,1039
0,accessories,61114.1160,16484.5983,1032
3,binders,55961.1130,16096.8016,1868
6,copiers,49749.2420,19327.2351,88
11,machines,42444.1220,-618.9264,147
4,bookcases,36004.1235,-1646.5117,306
1,appliances,30236.3360,8261.2699,492


In [239]:
# Cargar la tabla resumen en PostgreSQL
resumen_categoria.to_sql(
    name="dt_categoria_walmart",
    con=engine,
    schema="public",
    if_exists="replace",
    index=False
)

17

In [240]:
print("Tabla dt_categoria_walmart cargada correctamente")

Tabla dt_categoria_walmart cargada correctamente


In [241]:
# Relacionar las ventas con el resumen de cada categoría
consulta = """
SELECT
    v.order_id,
    v.customer_name,
    v.category,
    v.product_name,
    v.sales,
    v.profit,
    c.total_ventas_categoria,
    c.total_ganancia_categoria,
    c.unidades_vendidas_categoria
FROM public.ft_ventas_walmart v
INNER JOIN public.dt_categoria_walmart c
    ON v.category = c.category
ORDER BY v.sales DESC
LIMIT 20;
"""

In [242]:
ventas_categorias = pd.read_sql(consulta,engine)
ventas_categorias

,order_id,customer_name,category,product_name,sales,profit,total_ventas_categoria,total_ganancia_categoria,unidades_vendidas_categoria
0,CA-2014-140151,raymond buch,copiers,canon imageclass 2200 advanced copier,13999.960,6719.9808,49749.2420,19327.2351,88
1,CA-2011-143917,ken lonsdale,supplies,high speed automatic electric letter opener,8187.650,327.5060,18127.1220,626.0465,238
2,CA-2014-135909,jane waco,binders,fellowes pb500 electric punch plastic comb bin...,5083.960,1906.4850,55961.1130,16096.8016,1868
3,CA-2013-136301,edward hooks,supplies,high speed automatic electric letter opener,4912.590,196.5036,18127.1220,626.0465,238
4,CA-2014-149881,nick crebassa,machines,cubify cubex 3d printer double head print,4799.984,359.9988,42444.1220,-618.9264,147
5,CA-2013-138478,dennis pardue,binders,ibico epk-21 electric binding system,4535.976,1644.2913,55961.1130,16096.8016,1868
6,CA-2013-100300,max jones,machines,okidata mb760 printer,4476.800,503.6400,42444.1220,-618.9264,147
7,CA-2013-159016,karen ferguson,phones,apple iphone 5,4158.912,363.9048,98684.3520,9110.7426,1066
8,CA-2011-168494,nora preis,tables,bretford rectangular conference table tops,3610.848,135.4068,84754.5620,1482.6073,481
9,CA-2013-107104,maribeth schnelling,bookcases,dmi eclipse executive suite bookcases,3406.664,160.3136,36004.1235,-1646.5117,306


Interpretación de resultados

1.¿En qué estados se vende más y se obtienen mejores ganancias?
California es el estado con más ventas y también con la mayor ganancia, 
alcanzando aproximadamente 457.687 en ventas y 76.381 en ganancias. Washington ocupa el segundo lugar. 
En cambio, Arizona, Colorado y Oregon presentan ganancias negativas, 
por lo que en esos estados las ventas no están siendo igual de rentables.

2.¿Qué categorías se venden más y cuáles son las más rentables?
La categoría que genera el mayor monto de ventas es Chairs, seguida por Phones y Tables. 
Sin embargo, la categoría más rentable es Copiers, ya que presenta la ganancia total más alta. 
También se observa que Machines y Bookcases generan pérdidas, a pesar de registrar ventas.

3.¿Las ventas y ganancias han ido mejorando con el tiempo, o más bien han bajado?
Las ventas bajaron un poco entre 2011 y 2012. Sin embargo, desde 2013 empezaron a subir. 
El año 2014 fue el mejor, con ventas de aproximadamente 250.632 y ganancias de 43.900. 
En resumen, las cosas mejoraron en los últimos años que se analizaron.

4.¿Qué productos son los que más plata dejan?
El producto que genera el mayor monto de ventas es Canon imageCLASS 2200 Advanced Copier, 
con aproximadamente 13.999 en ventas. Además, también es el producto con la mayor ganancia dentro de los resultados, 
con cerca de 6.719.

5.¿Qué clientes son los que más gastan en total?
El cliente con el mayor monto acumulado de compras es Raymond Buch, con aproximadamente 14.345.
Luego aparecen Ken Lonsdale y Edward Hooks. 
Esto permite identificar a los clientes que generan una mayor cantidad de ingresos para la empresa.

La consulta actual muestra cuánto ha gastado cada cliente en total, no cuánto gasta en promedio por cada compra.

6.¿Cuántas ventas fueron rentables, generaron pérdidas o tuvieron ganancia neutra?
La mayoría de las ventas fueron rentables.
De los 3.203 registros que tenemos, 2.863 dieron ganancias. También hubo 318 ventas que dieron pérdidas.
Y 22 ventas que no dieron ganancias ni pérdidas.

7.¿Cuál es el tiempo promedio transcurrido entre la fecha del pedido y la fecha de envío?
En promedio, los pedidos demoraron aproximadamente 3,93 días en ser enviados, es decir, cerca de 4 días. 
El tiempo mínimo fue de 0 días y el máximo fue de 7 días.

El proceso ETL permitió limpiar, transformar y organizar la información de ventas de Walmart.
Las nuevas variables ayudaron a analizar los tiempos de envío, los márgenes de ganancia y la evolución de las ventas. Las consultas SQL permitieron identificar los estados, categorías, productos y clientes con mayor importancia para la empresa.
California presentó las mayores ventas y ganancias, mientras que el año 2014 obtuvo los mejores resultados del periodo analizado. Además, la mayoría de las ventas fueron rentables y el tiempo promedio de envío fue cercano a cuatro días.
Finalmente, la información quedó almacenada en PostgreSQL y fue exportada a archivos Excel y CSV para su posterior uso.

In [243]:
# Exportar dataset limpio
Walmart.to_excel("walmart_limpio.xlsx",index=False)

In [244]:

# Exportar tabla final
ventas_final.to_excel("ventas_final_walmart.xlsx",index=False)

In [245]:

# Exportar dataset analítico
dataset_analitico.to_excel("dataset_analitico_walmart.xlsx",index=False)

In [246]:

# Exportar resumen por categoría
resumen_categoria.to_excel("resumen_categoria_walmart.xlsx",index=False)

In [247]:
print("Archivos exportados correctamente")

Archivos exportados correctamente
